# OPERA — large-model run on Colab (A100)

Attention-free byte-level OPERA with the gated quaternion holographic memory, trained on
FineWeb-Edu bytes. Plan and results: `docs/OPERA_Recall_Research_2026-09-24.md` §8b–§8c.

**Order (run top to bottom; every step is idempotent and caches to Drive):**
1. Setup: GPU check, Drive, code, dependencies.
2. Correctness on this GPU: the Triton kernel test and the CUDA-vs-CPU model check. **Stop if either fails.**
3. Data: the Simple-Wikipedia test sets (for continuity with every earlier run), then the FineWeb-Edu pool.
4. Throughput benchmark → pick the model size.
5. Training: re-running the cell after a disconnect **resumes** from the last checkpoint on Drive.

Runtime → Change runtime type → **A100 GPU** (High-RAM on).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'no GPU: Runtime -> Change runtime type -> A100'
print(torch.__version__, torch.cuda.get_device_name(), 'bf16 native:', torch.cuda.get_device_capability()[0] >= 8)

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/opera_scale'      # everything that must survive a disconnect
RUNS = f'{DRIVE}/runs'                             # checkpoints, summaries, logs
DDATA = f'{DRIVE}/data'                            # cached datasets
LOCAL = '/content/data'                            # fast local copy used for training
REPO = '/content/opera-lm'
for d in (RUNS, DDATA, LOCAL):
    os.makedirs(d, exist_ok=True)
os.environ['OPERA_RUNS'] = RUNS

In [ ]:
# Code: the scale-colab branch. For a PRIVATE repo add a Colab secret GITHUB_TOKEN
# (key icon in the left bar; a fine-grained token with read access to OPERA-LM).
BRANCH = 'scale-colab'
try:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    tok = None
url = (f'https://{tok}@github.com/Merna-Khalid/OPERA-LM.git' if tok
       else 'https://github.com/Merna-Khalid/OPERA-LM.git')
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} {url} {REPO}
else:
    !git -C {REPO} pull -q
!git -C {REPO} log -1 --oneline
%cd {REPO}
!pip -q install datasets

## 2. Correctness on this GPU (stop if anything fails)

In [ ]:
!python -m opera_lm.triton_kernel --test
!python experiments/device_check.py --device cuda
!python experiments/device_check.py --device cuda --layers 4

## 3. Data

**3a. Simple-Wikipedia test sets** — the a20000 corpus (its test set is what every earlier BPB was
measured on) and the full-corpus pool test set. Rebuilt from the fixed `20231101.simple` snapshot and
checked against the counts measured locally; cached on Drive.

In [ ]:
import json, pickle, shutil
need = ['corpus_bytes_T1024_a20000.pkl', 'packed_bytes_T1024_all.test.pkl']
for f in need:
    if os.path.exists(f'{DDATA}/{f}') and not os.path.exists(f'assets/{f}'):
        shutil.copy(f'{DDATA}/{f}', f'assets/{f}')
if not os.path.exists('assets/corpus_bytes_T1024_a20000.pkl'):
    !python experiments/build_reprs.py --max-articles 20000 --modes bytes
if not os.path.exists('assets/packed_bytes_T1024_all.test.pkl'):
    !python experiments/build_packed.py
(tr, ts, tl, V), meta = pickle.load(open('assets/corpus_bytes_T1024_a20000.pkl', 'rb'))
assert (len(tr), len(ts), len(tl), sum(map(len, tr))) == (45765, 5092, 252, 37548997), 'a20000 corpus differs from the local build'
pm = json.load(open('assets/packed_bytes_T1024_all.meta.json')) if os.path.exists('assets/packed_bytes_T1024_all.meta.json') else None
if pm:
    assert (pm['n_seqs'], pm['total_tokens']) == (338339, 216919731), 'pool differs from the local build'
pt = pickle.load(open('assets/packed_bytes_T1024_all.test.pkl', 'rb'))['meta']
assert (pt['n_short_all'], pt['n_long']) == (37817, 1035), 'pool test set differs from the local build'
for f in need:
    if not os.path.exists(f'{DDATA}/{f}'):
        shutil.copy(f'assets/{f}', f'{DDATA}/{f}')
print('Simple-Wikipedia test sets OK')

**3b. FineWeb-Edu byte pool.** `MAX_BYTES` = training bytes in the pool. Size it from the benchmark
(tokens needed = steps × batch × 1024); 4 GB is enough for ~2 passes of a 2 B-byte run. Built once
(streams from the Hub, ~20–60 min for 4 GB), cached on Drive, copied to local disk for training.

In [ ]:
MAX_BYTES = 4e9
tag = f'fineweb_bytes_T1024_{MAX_BYTES / 1e9:g}GB'
FW = f'{LOCAL}/{tag}'
files = [f'{tag}.{s}' for s in ('tokens.npy', 'offsets.npy', 'test.pkl', 'meta.json')]
if all(os.path.exists(f'{DDATA}/{f}') for f in files):
    for f in files:
        if not os.path.exists(f'{LOCAL}/{f}'):
            shutil.copy(f'{DDATA}/{f}', f'{LOCAL}/{f}')
else:
    !python experiments/build_fineweb_bytes.py --max-bytes {MAX_BYTES} --out-dir {LOCAL}
    for f in files:
        shutil.copy(f'{LOCAL}/{f}', f'{DDATA}/{f}')
print(json.load(open(f'{FW}.meta.json')))

## 4. Throughput benchmark

One training step (forward + backward + Muon/AdamW, bf16, torch.compile) per size. The first call of
each size includes compilation. Results are appended to `runs/bench.jsonl`; paste the table back into
the conversation to pick the model.

In [ ]:
!python experiments/bench_scale.py --out {RUNS}/bench.jsonl \
    --grid 768x2x32 1024x2x32 1024x4x32 1536x2x32 1536x4x32 1024x8x32 2048x4x16 1536x4x64

## 5. Training

Arm names encode the model: `fw_d{width}_L{layers}` (memory slots = width/4; L > 2 uses the additive
residual with 1/√(2L) init). Tokens seen = `STEPS × BATCH × 1024`.

Re-run this cell after a disconnect: `--resume` continues the exact batch stream from the last
checkpoint (`--save-every`). At the end the model is scored on every test set (FineWeb-Edu, and the
Simple-Wikipedia a20000 / pool / new-articles sets).

In [ ]:
ARM, STEPS, BATCH = 'fw_d1024_L4', 20000, 32     # set from the benchmark
LOG = f'{RUNS}/{ARM}.log'
!python experiments/repr_study.py --arms {ARM} --steps {STEPS} --batch {BATCH} \
    --packed {FW} --test-pkl {FW}.test.pkl --pool-test assets/packed_bytes_T1024_all \
    --save-every 1000 --resume --device cuda 2>&1 | tee -a {LOG}

In [ ]:
# results so far
s = json.load(open(f'{RUNS}/repr_summary.json'))
for k, v in s.items():
    print(k, f"params {v['params']/1e6:.1f}M", f"FineWeb BPB {v['bpb']:.4f}", v.get('pool_eval'))